# Storage Layout and Logging

## What you'll learn

- The three-path model: delta, staging, and working directories
- What's inside the runs directory after a pipeline completes
- Delta Lake table structure and what each table contains
- Working directory defaults and when to preserve them
- Configuring logging: level, file handler, noise suppression

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Exploring Results](../01-getting-started/02-exploring-results.ipynb)  
**Estimated time:** 15 minutes  
**GPU required:** No.

---

Inspect a run’s durable results, staged worker output, execution workspaces,
and logs.

In [ ]:
from __future__ import annotations

import os
import tempfile

import polars as pl

from artisan.operations.examples import DataGenerator, DataTransformer, MetricCalculator
from artisan.orchestration import PipelineManager
from artisan.schemas import TablePath
from artisan.utils import configure_logging, tutorial_setup
from artisan.visualization import inspect_pipeline, inspect_step

In [ ]:
from functools import partial
from tempfile import TemporaryDirectory

# Keep this tutorial's stores and source files in one owned temporary workspace.
workspace = TemporaryDirectory(prefix="artisan-tutorial-")
tutorial_setup = partial(tutorial_setup, base_dir=workspace.name)

## Store, files, staging, and working roots

| Path | Purpose | Cleanup |
|---|---|---|
| `delta_root` | Accepted content, metadata, provenance, and selections | Retained |
| `files_root` | Managed external carriers | Retained with the store |
| `staging_root` | Worker evidence awaiting selected publication | Cleanup only after verified commitment unless preserved |
| `working_root` | Owned execution scratch | Cleanup after verified closure unless preserved |

Local stores default to a sibling `files` directory. Custom or remote stores
must bind their files root explicitly. Unselected staging is diagnostic evidence;
it does not authorize publication. Unknown command ownership retains scratch.

In [ ]:
env = tutorial_setup("storage_layout")

print(f"delta_root:   {env.delta_root}")
print(f"staging_root: {env.staging_root}")
print(f"working_root: {env.working_root}")
print(f"\nAll under: {env.runs_dir}")

## Running a pipeline and exploring the output

Run a three-step pipeline and inspect the directory tree afterward.

In [ ]:
pipeline = PipelineManager.create(
    name="storage_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 3, "seed": 42},
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
)
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
)

result = pipeline.finalize()
print(f"Pipeline complete: {result['total_steps']} steps")

In [ ]:
def show_tree(path: str, prefix: str = "", max_depth: int = 2, depth: int = 0) -> None:
    """Display a directory tree up to max_depth."""
    if depth >= max_depth or not os.path.isdir(path):
        return
    entries = sorted(os.path.join(path, name) for name in os.listdir(path))
    for i, entry in enumerate(entries):
        connector = (
            "\u2514\u2500\u2500 " if i == len(entries) - 1 else "\u251c\u2500\u2500 "
        )
        suffix = "/" if os.path.isdir(entry) else ""
        print(f"{prefix}{connector}{os.path.basename(entry)}{suffix}")
        if os.path.isdir(entry):
            extension = "    " if i == len(entries) - 1 else "\u2502   "
            show_tree(entry, prefix + extension, max_depth, depth + 1)


print(f"{os.path.basename(env.runs_dir)}/")
show_tree(env.runs_dir, max_depth=3)

For this successful run:

- `delta/` contains the durable Delta Lake tables.
- `staging/` has no remaining payloads after verified commitment.
- `working/` is empty after sandbox cleanup.
- `logs/runs/` contains a directory and `pipeline.log` for each manager session.

## Inspect the stored tables

The public artifact index records identities and types. Content tables hold each
artifact type’s data, provenance tables hold relationships, and orchestration
tables record steps and executions.

Print the framework table paths and see which were created by this run:

In [ ]:
for tp in TablePath:
    table_dir = os.path.join(env.delta_root, tp)
    exists = "\u2713" if os.path.exists(table_dir) else "\u2717"
    print(f"  {exists} {tp.value}")

`inspect_pipeline` and `inspect_step` read committed results. They apply
Artisan’s visibility rules so an incomplete persistence attempt does not appear
as accepted output:

In [ ]:
inspect_pipeline(env.delta_root)

In [ ]:
inspect_step(env.delta_root, step_number=0)

The query below reads authoritative artifact references without hydrating
content. Protected origin and current selection are separate: passing an
artifact through Filter does not rewrite its original creation step.

In [ ]:
from artisan.inspection import inspect_evidence

references = inspect_evidence(
    "query_artifacts",
    {"pipeline_run_id": pipeline.config.pipeline_run_id, "limit": 100},
    delta_root=env.delta_root,
)
artifact_index = pl.DataFrame([item.model_dump() for item in references.items])
artifact_index.select(
    "artifact_id", "artifact_type", "origin_step_number", "current_step_number"
)

## Working directory defaults

If you omit `working_root`, execution sandboxes use the system temp directory.

In [ ]:
print(f"Default working_root: {tempfile.gettempdir()}")
print(f"Tutorial working_root: {env.working_root}")

For debugging, `preserve_working=True` keeps owned scratch after verified
closure, and `preserve_staging=True` keeps committed worker evidence. Pass them
to `PipelineManager.create()`. Unknown ownership always retains affected evidence.
Resume can finish already selected plans; it cannot adopt arbitrary staged files.

## Configuring logging

A local pipeline manager owns a session log from creation through `finalize()`.
Read `pipeline.log_path` to find it, including after the run finishes. Resuming a
run creates a new session log.

`configure_logging` sets the process-wide severity and console behavior. Use
`level="DEBUG"` to include more detail; `level="INFO"` is enough for this example.

In [ ]:
# configure_logging is idempotent — safe to call multiple times
configure_logging(level="INFO", suppress_noise=True)

By default, `suppress_noise=True` sets the httpx, httpcore, and asyncio loggers
to `CRITICAL`, suppressing their debug output.

In [ ]:
# The owned path stays available after pipeline.finalize().
log_path = pipeline.log_path
print(f"Session log: {log_path}")
if log_path is not None:
    with open(log_path) as f:
        lines = f.read().splitlines()
    print(f"Pipeline log: {len(lines)} lines")
    print("\n".join(lines[-5:]))

The session log contains this manager’s Artisan orchestration messages. Worker
stdout and unrelated application logs have separate destinations. Finalize the
manager to flush and close its log. Cloud storage does not create this local
file, so `pipeline.log_path` is `None` there.

For failed executions, use `inspect_failures` to find their failure reports.
[Error Handling in Practice](../05-errors-and-control/02-error-visibility.ipynb)
shows how to read those reports.

## Selected-plan recovery

Stop the old driver before any repair write. Only one driver or repair process
may own the store. Read-only inspection reports selected incomplete publications,
unresolved ownership, and unclaimed evidence separately.

In [ ]:
print(
    "artisan store repair --delta-root runs/delta --files-root runs/files --staging-root runs/staging"
)

Report mode audits without writes. Add `--apply` to finish already selected
immutable publications, and `--preserve-staging` to keep their original evidence.
Repair does not choose a new winner, claim orphan seals, rerun science, or
abandon an accepted decision. Use the tooling for the original release to inspect
historical store formats; this writer does not relabel them.

## Next steps

- [Storage and Delta Lake](../../concepts/storage-and-delta-lake.md) — Storage and visibility
- [Configure S3-Compatible Storage](../../how-to-guides/configuring-s3.md) — Store results remotely
- [Error Handling in Practice](../05-errors-and-control/02-error-visibility.ipynb) — Read failure reports
- [Resume and Caching](../03-caching/01-resume-and-caching.ipynb) — Reuse persisted work